### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="nci_pancreatic_cancer",
    dataset_year="2003",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="GOV Website",
    original_dataset_source_download_link="https://home.ccr.cancer.gov/ncifdaproteomics/ppatterns.asp",
    download_description="""
We download the data to a predefined folder.

mkdir -p local-data-warehouse/nci_pancreatic_cancer && \
wget -P local-data-warehouse/nci_pancreatic_cancer https://home.ccr.cancer.gov/ncifdaproteomics/CancerCellPanINDataBinned.zip && \
unzip local-data-warehouse/nci_pancreatic_cancer/CancerCellPanINDataBinned.zip -d local-data-warehouse/nci_pancreatic_cancer/ && \
rm local-data-warehouse/nci_pancreatic_cancer/CancerCellPanINDataBinned.zip && \
mv local-data-warehouse/nci_pancreatic_cancer/CancerCellPanINDataBinned/02-Control/PanINCTL local-data-warehouse/nci_pancreatic_cancer/ && \
mv local-data-warehouse/nci_pancreatic_cancer/CancerCellPanINDataBinned/03-PanIN local-data-warehouse/nci_pancreatic_cancer && \
rm -r local-data-warehouse/nci_pancreatic_cancer/CancerCellPanINDataBinned
""",
    # References
    academic_reference_bibtex="""@article{Hingorani2003PreinvasiveAI,
  title={Preinvasive and invasive ductal pancreatic cancer and its early detection in the mouse},
  author={Sunil R. Hingorani and Emanuel F. Petricoin and Anirban Maitra and Vinodh Rajapakse and Cynthia King and Michael A. Jacobetz and Sarah Ross and Thomas P. Conrads and Timothy D. Veenstra and Ben A. Hitt and Yoshiya Kawaguchi and David Johann and Lance A. Liotta and Howard C. Crawford and Mary E. Putt and Tyler Jacks and Christopher V. Wright and Ralph H. Hruban and Andrew M. Lowy and David A. Tuveson},
  journal={Cancer Cell},
  year={2003},
  volume={4},
  number={6},
  pages={437-450},
  doi={10.1016/s1535-6108(03)00309-x},
  url={https://doi.org/10.1016/s1535-6108(03)00309-x}
}
""",
    academic_reference_bibtex_key="Hingorani2003PreinvasiveAI",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
        - We merge all samples located in the .csv files into one DataFrame.
        - We label the samples based on name of the folder they are located in: "PanINCTL" translates to healthy patient cases while "03-PanIN" to individuals with a pancreatic cancer.
        - We shuffle the data since it is concatenated in the folders order.
    """,
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="HasCancer",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="HasCancer",
)

## Preprocessing

In [2]:
import os
import pandas as pd

valid_labels = {
    "PanINCTL",
    "03-PanIN"
}

def csv_to_dataframe(file, label):
    """
    Convert each sample from the .csv file to a DataFrame with a label based on the name of the folder it is located in. 
    """
    label_map = {
        "PanINCTL": "No",
        "03-PanIN": "Yes",
    }

    df = pd.read_csv(file).T
    df.reset_index(drop=True, inplace=True)
    df.drop(0, inplace=True)

    target_feature = "HasCancer"
    df.columns = [f"Attribute {i+1}" for i in range(len(df.columns) - 1)] + [target_feature]
    df[target_feature] = label_map.get(label, label)

    return df

r = {}
subdirs = [dirs[0] for dirs in os.walk(dataset_mold.path)][1:]
for subdir in subdirs:
    label = subdir.split("/")[-1]
    _, _, files = next(os.walk(subdir))
    r[label] = [os.path.join(subdir, name) for name in files]

df = pd.concat(
        [csv_to_dataframe(filepath, label) for label in r if label in valid_labels for filepath in r[label]], 
        ignore_index=True
    )
df["HasCancer"] = df["HasCancer"].astype("category")
df = df.sample(frac=1, random_state=99).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 181
Columns: 6771
Use sampling: False (sample size: 181)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Attribute 5822', 'Attribute 4084', 'Attribute 6016', 'Attribute 5797', 'Attribute 5818', 'Attribute 5806', 'Attribute 5812', 'Attribute 5783', 'Attribute 5784', 'Attribute 5817']
Rows remaining as candidates after top-10 filter: 0 (of 181)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Attribute 1,Attribute 2,Attribute 3,Attribute 4,Attribute 5,Attribute 6,Attribute 7,Attribute 8,Attribute 9,Attribute 10,Attribute 11,Attribute 12,Attribute 13,Attribute 14,Attribute 15,Attribute 16,Attribute 17,Attribute 18,Attribute 19,Attribute 20,Attribute 21,Attribute 22,Attribute 23,Attribute 24,Attribute 25,Attribute 26,Attribute 27,Attribute 28,Attribute 29,Attribute 30,Attribute 31,Attribute 32,Attribute 33,Attribute 34,Attribute 35,Attribute 36,Attribute 37,Attribute 38,Attribute 39,Attribute 40,Attribute 41,Attribute 42,Attribute 43,Attribute 44,Attribute 45,Attribute 46,Attribute 47,Attribute 48,Attribute 49,Attribute 50,Attribute 51,Attribute 52,Attribute 53,Attribute 54,Attribute 55,Attribute 56,Attribute 57,Attribute 58,Attribute 59,Attribute 60,Attribute 61,Attribute 62,Attribute 63,Attribute 64,Attribute 65,Attribute 66,Attribute 67,Attribute 68,Attribute 69,Attribute 70,Attribute 71,Attribute 72,Attribute 73,Attribute 74,Attribute 75,Attribute 76,Attribute 77,Attribute 78,Attribute 79,Attribute 80,Attribute 81,Attribute 82,Attribute 83,Attribute 84,Attribute 85,Attribute 86,Attribute 87,Attribute 88,Attribute 89,Attribute 90,Attribute 91,Attribute 92,Attribute 93,Attribute 94,Attribute 95,Attribute 96,Attribute 97,Attribute 98,Attribute 99,Attribute 100,Attribute 101,Attribute 102,Attribute 103,Attribute 104,Attribute 105,Attribute 106,Attribute 107,Attribute 108,Attribute 109,Attribute 110,Attribute 111,Attribute 112,Attribute 113,Attribute 114,Attribute 115,Attribute 116,Attribute 117,Attribute 118,Attribute 119,Attribute 120,Attribute 121,Attribute 122,Attribute 123,Attribute 124,Attribute 125,Attribute 126,Attribute 127,Attribute 128,Attribute 129,Attribute 130,Attribute 131,Attribute 132,Attribute 133,Attribute 134,Attribute 135,Attribute 136,Attribute 137,Attribute 138,Attribute 139,Attribute 140,Attribute 141,Attribute 142,Attribute 143,Attribute 144,Attribute 145,Attribute 146,Attribute 147,Attribute 148,Attribute 149,Attribute 150,Attribute 151,Attribute 152,Attribute 153,Attribute 154,Attribute 155,Attribute 156,Attribute 157,Attribute 158,Attribute 159,Attribute 160,Attribute 161,Attribute 162,Attribute 163,Attribute 164,Attribute 165,Attribute 166,Attribute 167,Attribute 168,Attribute 169,Attribute 170,Attribute 171,Attribute 172,Attribute 173,Attribute 174,Attribute 175,Attribute 176,Attribute 177,Attribute 178,Attribute 179,Attribute 180,Attribute 181,Attribute 182,Attribute 183,Attribute 184,Attribute 185,Attribute 186,Attribute 187,Attribute 188,Attribute 189,Attribute 190,Attribute 191,Attribute 192,Attribute 193,Attribute 194,Attribute 195,Attribute 196,Attribute 197,Attribute 198,Attribute 199,Attribute 200,Attribute 201,Attribute 202,Attribute 203,Attribute 204,Attribute 205,Attribute 206,Attribute 207,Attribute 208,Attribute 209,Attribute 210,Attribute 211,Attribute 212,Attribute 213,Attribute 214,Attribute 215,Attribute 216,Attribute 217,Attribute 218,Attribute 219,Attribute 220,Attribute 221,Attribute 222,Attribute 223,Attribute 224,Attribute 225,Attribute 226,Attribute 227,Attribute 228,Attribute 229,Attribute 230,Attribute 231,Attribute 232,Attribute 233,Attribute 234,Attribute 235,Attribute 236,Attribute 237,Attribute 238,Attribute 239,Attribute 240,Attribute 241,Attribute 242,Attribute 243,Attribute 244,Attribute 245,Attribute 246,Attribute 247,Attribute 248,Attribute 249,Attribute 250,Attribute 251,Attribute 252,Attribute 253,Attribute 254,Attribute 255,Attribute 256,Attribute 257,Attribute 258,Attribute 259,Attribute 260,Attribute 261,Attribute 262,Attribute 263,Attribute 264,Attribute 265,Attribute 266,Attribute 267,Attribute 268,Attribute 269,Attribute 270,Attribute 271,Attribute 272,Attribute 273,Attribute 274,Attribute 275,Attribute 276,Attribute 277,Attribute 278,Attribute 279,Attribute 280,Attribute 281,Attribute 282,Attribute 283,Attribute 284,Attribute 285,Attribute 286,Attribute 287,Attribute 288,Attribute 289,Attribute 290,Attribute 291,Attribute 292,Attribute 293,Attri

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,HasCancer,category,0.0,0.0,2.0,"No, Yes"
1,Attribute 1,float64,0.0,0.0,151.0,"96.0, 321.0, 141.0, 233.0, 221.0, 182.0, 105.0, 187.0, 151.0, 134.0"
2,Attribute 2,float64,0.0,0.0,151.0,"313.0, 165.0, 404.0, 572.0, 172.0, 234.0, 273.0, 463.0, 671.0, 395.0"
3,Attribute 3,float64,0.0,0.0,137.0,"61.0, 50.0, 139.0, 226.0, 105.0, 163.0, 73.0, 43.0, 202.0, 270.0"
4,Attribute 4,float64,0.0,0.0,156.0,"171.0, 259.0, 306.0, 453.0, 205.0, 561.0, 769.0, 518.0, 475.0, 699.0"
5,Attribute 5,float64,0.0,0.0,169.0,"716.0, 340.0, 234.0, 481.0, 295.0, 683.0, 413.0, 642.0, 250.0, 300.0"
6,Attribute 6,float64,0.0,0.0,155.0,"77.0, 139.0, 316.0, 67.0, 78.0, 434.0, 148.0, 43.0, 85.0, 134.0"
7,Attribute 7,float64,0.0,0.0,149.0,"342.0, 146.0, 293.0, 134.0, 111.0, 442.0, 119.0, 129.0, 154.0, 144.0"
8,Attribute 8,float64,0.0,0.0,162.0,"518.0, 206.0, 203.0, 264.0, 658.0, 284.0, 342.0, 442.0, 249.0, 152.0"
9,Attribute 9,float64,0.0,0.0,151.0,"72.0, 62.0, 79.0, 164.0, 179.0, 148.0, 139.0, 96.0, 321.0, 249.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Attribute 1,181.0,271.823204,173.239628,42.0,952.0
Attribute 2,181.0,330.320442,162.949471,66.0,709.0
Attribute 3,181.0,154.635359,88.926247,23.0,358.0
Attribute 4,181.0,426.602210,295.474979,57.0,1549.0
Attribute 5,181.0,629.889503,421.679247,99.0,2460.0
Attribute 6,181.0,224.370166,149.471294,28.0,926.0
Attribute 7,181.0,252.767956,152.178555,47.0,721.0
Attribute 8,181.0,415.635359,230.182801,79.0,1157.0
Attribute 9,181.0,242.160221,156.342579,29.0,791.0
Attribute 10,181.0,510.906077,396.665598,52.0,1909.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count   pct
column    rank                   
HasCancer 1       No    101  55.8
          2      Yes     80  44.2

In [8]:
# Target Distribution
target_df

,count,pct
HasCancer,,
No,101,55.8
Yes,80,44.2


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to nci_pancreatic_cancer/019d9d23-4ace-71c9-b3c3-06491168b970
019d9d23-4ace-71c9-b3c3-06491168b970
7dcf4c9d989f18d89e79fd5cb39207664e5bb05a0b1dd6c2fece483bb052ded9
